# Lesson 2: NumPy

One operation on a whole column of numbers.

<img src="../img/map-2.svg" width="85%" alt="">

## The cost of a loop

Project 1 read 50,000 events, and you waited a moment. The full sample here is 300,000 events, and an LHC experiment records billions.

Every analysis asks the same question of every event, so the loop is always the same shape. Array programming hands that loop to fast compiled code: you write the operation once, the library applies it to the whole column.

NumPy is the shared language of scientific Python: pandas, hist, SciPy and Awkward Array all build on its arrays. Learn the moves once, use them everywhere.

## Loops and arrays

A loop handles one value at a time. An array handles the whole column.

<img src="../img/loop-vs-column.svg" width="85%" alt="">

## Programming paradigms

Paradigms are rough classifications of programming styles.

| Paradigm | The style | Where you meet it |
|:--|:--|:--|
| imperative | spell out every step, one element at a time | project 1 |
| object-oriented | bundle data with the methods that use it | lesson 3 |
| functional | pass functions to functions | `map`, comprehensions |
| array-oriented | one operation on whole arrays | this lesson |

The same job four ways: square every element.

**Imperatively**, element by element:

In [ ]:
import numpy as np

input_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
output_data = np.zeros(9, dtype=int)

for i in range(len(input_data)):
    output_data[i] = input_data[i]**2

output_data

**Functionally**: `map` from lesson 1 calls `square` on every element, `np.fromiter` collects the results into an array — or the comprehension shorthand:

In [ ]:
def square(x):
    return x**2

np.fromiter(map(square, input_data), int)

In [ ]:
np.asarray([x**2 for x in input_data])

**Array-oriented**:

In [ ]:
input_data**2

Functional code still thinks in elements: the `x` in `square(x)` is one number. Array-oriented code has no element in sight: the variables are whole columns, and the question is how the *distribution* changes.

Every array language, from APL in the 1960s to MATLAB, R and NumPy, was built for the same job: interactive data analysis. Short questions, quick answers, at any data size.

Paradigms are styles, not laws: Python mixes them freely.

You are living the comparison this week: project 1 was the imperative Higgs search, and project 2 is the same search, array-oriented.

## Arrays

An array is one block of values that all have the same type.

Everyone imports NumPy under the short name `np`.

<img src="../img/list-vs-array.svg" width="80%" alt="">

In [ ]:
import numpy as np

pt = np.array([63.0, 38.1, 4.0, 54.3])
pt.dtype, pt.shape

## Making arrays

From a list, from a range, from nothing.

In [ ]:
np.arange(5), np.linspace(0.0, 1.0, 5)

In [ ]:
np.zeros(4), np.ones(3)

`reshape` rearranges the same values into a new shape — nothing is copied:

In [ ]:
np.arange(6).reshape(2, 3)

## Every array has a `dtype`

One type for the whole block, and the operations follow it. `astype` converts the whole column at once.

In [ ]:
counts = np.array([5, 2, 9])
counts.dtype, (counts / 2).dtype

In [ ]:
mask = np.array([True, False, True])
mask.astype(np.int32)

## Why arrays are fast

Python never knows what a value is until it looks. Before every single `*`, it asks: what type is this, and what does multiplying it mean?

For one number, that cost is invisible. In a loop over a million list items, the asking happens a million times — the questions are the slow part, not the arithmetic.

An array is a promise: one dtype, values side by side. The question is asked once, and compiled code sweeps the whole column.

<img src="../img/array-speed.svg" width="90%" alt="">

Memory is the dtype times the count, nothing more. `astype` trades precision for half the space:

In [ ]:
big = np.ones(1_000_000)
big.nbytes, big.astype(np.float32).nbytes

## Quizlet 2.1 — who shares the memory?

Guess before you run: after the last line, what is in `a`?

<img src="../img/view.svg" width="70%" alt="">

In [ ]:
a = np.array([63.0, 38.1, 4.0])
b = a[:2]
b[0] = 999.0
a

A slice of an array is a *view*: the same memory under a new name. Fast, because nothing is copied — and a surprise, until you know it. `a[:2].copy()` gives you your own block. (A slice of a *list* is a copy. Python keeps you alert.)

## Arithmetic

Arithmetic applies to every element at once.

<img src="../img/elementwise.svg" width="70%" alt="">

In [ ]:
pt * 2

## Broadcasting

Shapes do not have to match: a single number is reused for every element, and a row and a column spread out into a table.

<img src="../img/broadcast.svg" width="80%" alt="">


In [ ]:
pt - 20.0

In [ ]:
pt2d = np.array([[63.0, 38.1, 4.0],
                 [54.3, 23.5, 52.9]])
scale = np.array([1.0, 1.0, 0.5])

pt2d * scale

## Lists are different

A list does something else with `* 2`: it repeats itself.

In [ ]:
[63.0, 38.1, 4.0, 54.3] * 2

## NumPy functions

NumPy functions apply to every element. Arrays of the same shape combine element by element.

The momentum along the beam is $p_z = p_T \sinh\eta$.

<img src="../img/elementwise-two.svg" width="70%" alt="">

In [ ]:
eta = np.array([-0.72, -0.88, -0.32, -1.06])
pz = pt * np.sinh(eta)
pz

This holds for *any* closed-form formula: write it once with scalars in mind, hand it whole arrays, and every event is computed in one sweep. Your project-1 functions become project-2 functions by swapping `math` for `np`.

## Quizlet 2.4 — the mass that is not there

In project 1, `mass()` took a square root. What if the number under it is negative?

Guess before you run: an error, or something else?

<img src="../img/nan-flow.svg" width="85%" alt="">


In [ ]:
np.sqrt(np.array([4.0, -1.0]))

`nan` — *not a number* — with a warning, not an error. It flows silently through every later step, and `nan == nan` is `False`. When a mass distribution has a hole, look for a `nan` upstream.

## Speed

The array version is much faster than the loop.

`%timeit` runs a line many times and reports the average time.

In [ ]:
big = np.arange(1_000_000)
%timeit [x * 2 for x in big]

In [ ]:
%timeit big * 2

## Masks

A comparison on an array gives a mask: one `True` or `False` per element.

In [ ]:
mask = pt > 20
mask

## Selecting with a mask

A mask inside square brackets keeps the `True` entries.

<img src="../img/mask.svg" width="70%" alt="">

In [ ]:
pt[mask]

## Indexing with positions

Square brackets also accept a list of positions: any order, repeats allowed. A mask asks "which pass?", positions say "these, in this order".

In [ ]:
pt[np.array([2, 0, 0])]

## Masks across arrays

A mask made from one array can select from another.

Position 0 in `pt` and position 0 in `eta` describe the same muon.

<img src="../img/mask-other.svg" width="70%" alt="">

In [ ]:
eta[mask]

Indexing with positions turns coded data back into readable data. `np.unique` splits a text into its vocabulary and the position of each word in it:

In [ ]:
text = """Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do: once or twice she had peeped into the book her sister
was reading, but it had no pictures or conversations in it, and what is the use of
a book, thought Alice, without pictures or conversations?"""

words = np.array(text.replace(",", " ").replace(".", " ").replace("?", " ").replace("!", " ").replace("-", " ").split())
dictionary, index = np.unique(words, return_inverse=True)

In [ ]:
dictionary

In [ ]:
index

**Quizlet 2.8** — What's going to happen?

In [ ]:
dictionary[index]

<details>
    <summary><b>Hint...</b></summary>

```
index             : positions in corpus (0, 1, 2, ...) → integer codes
dictionary        : integer codes                      → words

dictionary[index] : positions in corpus (0, 1, 2, ...) → words
```

</details>

## Combining masks

Combine masks with `&` (and) and `|` (or). Put each condition in parentheses.

In [ ]:
(pt > 20) & (np.abs(eta) < 1.0)

## What does not work

The words `and` and `or` do not work on arrays, and neither does `12 < x < 120`.

Read the last line of the error.

In [ ]:
(pt > 20) and (np.abs(eta) < 1.0)

## Counting

Counting the `True` values counts what passes.

`True` counts as 1 and `False` as 0. The mean of a mask is the fraction that passes.

In [ ]:
print(mask.sum(), mask.mean())

## Reductions

A reduction turns many values into one.

<img src="../img/reduce.svg" width="75%" alt="">

In [ ]:
print(pt.max(), pt.min(), len(pt))

## Which one, not how big

`max` says how big. `argmax` says *where*. The position then looks up anything else about the winner.

<img src="../img/argmax.svg" width="80%" alt="">


In [ ]:
i = np.argmax(pt)
i, eta[i]

## `np.minimum` and `np.maximum`

`np.minimum` and `np.maximum` compare two arrays element by element.

In [ ]:
other = np.array([50.0, 20.0, 10.0, 60.0])
np.minimum(pt, other)

## Two dimensions

An array can have two dimensions. `axis` names the dimension that a reduction removes.

Here each row is an event. `axis=1` leaves one value per event.

<img src="../img/axis.svg" width="70%" alt="">

In [ ]:
pt2d = np.array([[63.0, 38.1, 4.0],
                 [54.3, 23.5, 52.9]])
pt2d.sum(axis=1)

`[row, column]` picks one value, and `:` means the whole axis. `pt2d[:, 0]` is the first column: one value per row.

In [ ]:
pt2d[0, 2], pt2d[:, 0]

## Invent data to test with

A generator with a seed gives the same "random" numbers on every run, so a test you write today fails the same way tomorrow.

In [ ]:
rng = np.random.default_rng(1)
rng.normal(91.2, 2.5, 5)

## Histograms

The `hist` library builds, fills and plots a histogram.

`Reg(50, 0, 100)` means 50 bins of equal width from 0 to 100.

<img src="../img/histogram.svg" width="65%" alt="">

In [ ]:
import hist

h = hist.Hist.new.Reg(50, 0, 100, label="pT [GeV]").Double()
h.fill(np.random.default_rng(1).exponential(20, 10_000))
h.plot();

## Quizlet 2.9 — guess the shape

Array-oriented languages bring data *distributions* to the foreground.

In [ ]:
from hist import Hist  # histogram library

Given a large dataset...

In [ ]:
dataset = np.random.normal(0, 1, 1000000)  # one MILLION data points

How are the data _distributed_?

In [ ]:
Hist.new.Reg(100, -5, 5).Double().fill(dataset)

What happens if we apply a function to _all values in the distribution_?

In [ ]:
dataset2 = dataset**2

In [ ]:
Hist.new.Reg(100, -1, 10).Double().fill(dataset2)

**Quizlet 2.9** — Can anyone guess what this distribution will look like?

In [ ]:
dataset3 = np.sin(1/dataset2)

(I can't.)

In [ ]:
Hist.new.Reg(100, -1, 1).Double().fill(dataset3)

## The NumPy toolbox

Everything in this lesson, on one slide. The puzzles ahead combine these and nothing else.

| Tool | What it does |
|:--|:--|
| elementwise operations | one formula, every element |
| broadcasting | a number, or a smaller shape, reused |
| slicing | ranges, steps, several dimensions |
| masks | keep what passes a condition |
| integer arrays | pick by position, any order |
| `reshape` | the same values, a new shape |
| reductions | many values to one, or one per row |

## How to think in arrays

The best way to learn is by solving problems.
Quizlets 2.11 to 2.16, roughly in rising order. Try each one in the workbook before its solution slide.

### Quizlet 2.11 — hit the target

`m` is 5 × 6. Find the single slice `m[?, ?]` that gives

```python
[[ 4,  5],
 [28, 29]]
```

In [ ]:
m = np.arange(30).reshape(5, 6)
m

#### Solution to quizlet 2.11

Rows in steps of four, columns from four onward.

In [ ]:
m[::4, 4:]

### Quizlet 2.12 — the spaces between

Compute the size of the spaces between consecutive elements.

<img src="../img/flat-operation.svg" width="40%" alt=""> <img src="../img/shifted-operation.svg" width="40%" alt="">

In [ ]:
array = np.array([1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8, 9.9])

#### Solution to quizlet 2.12

A shifted slice against an unshifted one — or the function NumPy compiled in.

In [ ]:
array[1:] - array[:-1]

In [ ]:
np.diff(array)

### Quizlet 2.13 — the length of a curve

`plt.plot` draws one array against another. How long is this curve?

<img src="../img/length-by-segment.svg" width="50%" alt="">

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, 2 * np.pi, 10000)
x = np.sin(3 * t)
y = np.sin(4 * t)

plt.plot(x, y);

#### Solution to quizlet 2.13

Every step at once, Pythagoras on the steps, one sum.

In [ ]:
np.sum(np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2))

### Quizlet 2.14 — slice in three dimensions

You can select items of a 3D array,

<img src="../img/array3d-highlight1.svg" width="25%" alt="">

with `array3d[:, 1:, 1:]`. Write the slice that selects these elements:

<img src="../img/array3d-highlight2.svg" width="25%" alt="">

In [ ]:
array3d = np.arange(2 * 3 * 5).reshape(2, 3, 5)
array3d

#### Solution to quizlet 2.14

Keep the first dimension (length 1), or reduce it away. Both are right.

In [ ]:
array3d[:1, :, 2:]

In [ ]:
array3d[0, :, 2:]

### Quizlet 2.15 — shrink a photograph

Scale this image down by a factor of 64 on both sides, using only [np.reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html), [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html), and [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

In [ ]:
import matplotlib.image

image = matplotlib.image.imread("../data/sun-shines-in-CMS.jpg")
image.shape

In [ ]:
plt.imshow(image);

1920 rows, 2560 columns, and the third axis is for (red, green, blue), all `np.uint8`.

Your strategy should be to reshape the array, such that the dimension of length `1920` becomes two new dimensions of length `1920 // 64` and `64` and the dimension of length `2560` becomes two new dimensions of length `2560 // 64` and `64`. Then average over each of the dimensions of length `64`.

The shape should change as

$$\left(1920, 2560, 3\right) \to \left(\frac{1920}{64}, 64, \frac{2560}{64}, 64, 3\right) \to \left(\frac{1920}{64}, \frac{2560}{64}, 3\right)$$

and then you need to turn the floating-point dtype back into unsigned 8-bit integers with [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

#### Solution to quizlet 2.15

In NumPy, the shapes of arrays are fluid. An array can be reshaped from 3 dimensional to 5 dimensional as long as the number of items is unchanged.

Reshaping provides new axes for reducers like [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) to apply.

Be sure to average over axis 3 before axis 1, since reduction removes an axis, changing the numbering of later axes.

In [ ]:
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=3,
    ),
    axis=1,
)

plt.imshow(resampled.astype(np.uint8));

Or alternatively, use negative axis numbers and do it in the other order.

In [ ]:
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=-4,
    ),
    axis=-2,
)

plt.imshow(resampled.astype(np.uint8));

Or as a third option, use `keepdims=True` to keep the axis numbering from changing and only remove the length 1 dimensions after all reductions are done.

In [ ]:
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=1,
        keepdims=True,
    ),
    axis=3,
    keepdims=True,
).reshape(1920 // 64, 2560 // 64, 3)

plt.imshow(resampled.astype(np.uint8));

### Quizlet 2.16 — the bytes are backwards

ROOT files store their floats big-endian; your machine is little-endian. Viewed naively, five ordinary numbers look broken:

<img src="../img/big-little-endian.svg" width="75%" alt="">

In [ ]:
raw = np.arange(5, dtype=">f4").tobytes()
as_bytes = np.frombuffer(raw, np.uint8)
as_bytes.view(np.float32)

Fix it: reverse the bytes *in groups of four*, using only `reshape` and a slice.

#### Solution to quizlet 2.16

A short-lived axis expresses "groups of four", and a negative-step slice reverses each group.

In [ ]:
as_bytes.reshape(-1, 4)[:, ::-1].reshape(-1).view(np.float32)

## What you can now read

| You see | It means |
|---|---|
| `np.array([...])` | make an array |
| `pt * 2`, `np.sqrt(pt)` | apply to every element |
| `pt > 20` | a mask of `True` and `False` |
| `pt[mask]` | keep the `True` entries |
| `(a > 1) & (b < 2)` | both conditions |
| `mask.sum()` | how many pass |
| `np.minimum(a, b)` | the smaller of two arrays, element by element |
| `pt - 20.0` | the number is reused for every element |
| `pt[np.array([2, 0])]` | pick by position |
| `np.arange(5)`, `np.zeros(4)` | make an array from nothing |
| `x.astype(np.int32)` | convert the whole column |
| `pt2d[:, 0]` | the first column |
| `np.argmax(pt)` | the position of the biggest |


## Now: project 2.

[project-higgs.ipynb](project-higgs.ipynb): the same analysis on whole columns.

<img src="../img/lepton-columns.svg" width="65%" alt="">